<a href="https://colab.research.google.com/github/zain4cs/NLP_Project/blob/main/Emotions_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [32]:
df = pd.read_csv("/content/drive/MyDrive/Datasets/train.txt", sep = ';', header=None, names = ['text', 'emotion'])
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [33]:
df.isnull().sum()

,0
text,0
emotion,0


_______________________

**Convert Into Numbers**

In [34]:
unique_emotions = df['emotion'].unique()

In [35]:
emotions_numbers = {}
i = 0
for emo in unique_emotions:
  emotions_numbers[emo] = i
  i += 1

df['emotion'] = df['emotion'].map(emotions_numbers)

In [36]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


**1: Lower Case**

In [37]:
df['text'] = df['text'].apply(lambda x: x.lower())

**2: Removing Punctuation**

In [38]:
import string

def remove_punc(txt):
  return txt.translate(str.maketrans('', '', string.punctuation))

In [39]:
df['text'] = df['text'].apply(remove_punc)

**3: Remove Numbers**

In [40]:
def remove_numbers(txt):
  new = ""
  for i in txt:
    if not i.isdigit():
      new = new + i

  return new

df['text'] = df['text'].apply(remove_numbers)

**4: Remove URL/Links**

**5: Remove Emojis**

In [41]:
def  remove_emojis(txt):
  new = ""
  for i in txt:
    if i.isascii():
      new += i
  return new


df['text'] = df['text'].apply(remove_emojis)

**6: Remove Stopwords**

In [42]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [43]:
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [44]:
stop_words = set(stopwords.words('english'))
len(stop_words)

198

In [45]:
# Before
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [46]:
def remove(txt):
  # words = word_tokenize(txt)
  words = txt.split()

  cleaned = []
  for i in words:
      if not i in stop_words:
        cleaned.append(i)

  return ' '.join(cleaned)

In [47]:
df['text'] = df['text'].apply(remove)

In [48]:
# After
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

________________

**Bag Of Words**

In [49]:
from sklearn.feature_extraction.text import CountVectorizer

doocuments = [
    "I love pizza",
    "Pizza is the best",
    "I love pasta",
    "Pasta is greate"
]

vectorizer = CountVectorizer(ngram_range=(2,2))

X = vectorizer.fit_transform(doocuments)

print("Vucabulary", vectorizer.get_feature_names_out())
print("\nBOW Matrix\n", X.toarray())

Vucabulary ['is greate' 'is the' 'love pasta' 'love pizza' 'pasta is' 'pizza is'
 'the best']

BOW Matrix
 [[0 0 0 1 0 0 0]
 [0 1 0 0 0 1 1]
 [0 0 1 0 0 0 0]
 [1 0 0 0 1 0 0]]


_________________

**TF-IDF**


---


*   TF:  Term Frequency
*   IDF: Inverse Document Frequency



In [50]:
from sklearn.feature_extraction.text import TfidfVectorizer

documents = [
    "I love pizza",
    "Pizza is the best",
    "I love pasta",
    "Pasta is greate"
]

vectorizer = TfidfVectorizer()
X  = vectorizer.fit_transform(documents)

print("Vucabulary", vectorizer.get_feature_names_out())
print("\nBOW Matrix\n", X.toarray())

Vucabulary ['best' 'greate' 'is' 'love' 'pasta' 'pizza' 'the']

BOW Matrix
 [[0.         0.         0.         0.70710678 0.         0.70710678
  0.        ]
 [0.55528266 0.         0.43779123 0.         0.         0.43779123
  0.55528266]
 [0.         0.         0.         0.70710678 0.70710678 0.
  0.        ]
 [0.         0.66767854 0.52640543 0.         0.52640543 0.
  0.        ]]


______________

**Train Model**

In [51]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.2, random_state=42)

In [52]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

bow_vectorizer = CountVectorizer()
# tfidf_vectorizer = TfidfVector()

X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

**Model:**

In [53]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [54]:
nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)

MultinomialNB()

In [55]:
pred_nb = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_nb))

0.768125


In [56]:
pred_nb

array([0, 5, 0, ..., 5, 5, 0])

In [57]:
y_test

,emotion
8756,0
4660,5
6095,0
304,5
8241,0
...,...
15578,5
5746,5
6395,5
7624,5


_________________

**Now Use TF-IDF_Vectorizer**

In [58]:
tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf, y_train)

pred_nb2 = nb2_model.predict(X_test_tfidf)
print(accuracy_score(y_test, pred_nb2))


0.6609375


**Logistics Regression**

In [60]:
from sklearn.linear_model import LogisticRegression

In [61]:
tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)

log_pred = lr_model.predict(X_test_tfidf)
print(accuracy_score(y_test, log_pred))


0.8628125
